# Chapitre 4 — Architectures et biais inductifs

Atelier corrigé : NumPy + Matplotlib ; données synthétiques locales.
Annoncer le domaine circulaire et distinguer le test IID des positions nouvelles.


## Étape 1
Consigner les versions.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
print('NumPy :', np.__version__)



## Étape 2
Démontrer l’équivariance avant les assertions.


In [ ]:
# CELL 2 — Corrélation circulaire ; convention p+j, sans retournement du noyau.
def circular_correlation(X, kernel):
    # X : (B,L) ; kernel : (K,) ; positions modulo L.
    return sum(kernel[j]*np.roll(X,-j,axis=1) for j in range(len(kernel)))

r=np.random.default_rng(1)
X=r.normal(size=(4,16));kernel=np.array([1.,0.,-1.])
for delta in [1,2,5]:
    left=circular_correlation(np.roll(X,delta,axis=1),kernel)
    right=np.roll(circular_correlation(X,kernel),delta,axis=1)
    np.testing.assert_allclose(left,right,atol=1e-12)
base=circular_correlation(X,kernel)[:,::2]
even=circular_correlation(np.roll(X,2,axis=1),kernel)[:,::2]
np.testing.assert_allclose(even,np.roll(base,1,axis=1),atol=1e-12)
odd=circular_correlation(np.roll(X,1,axis=1),kernel)[:,::2]
print('Écart déplacement impair / sortie non déplacée après stride 2 :',np.max(np.abs(odd-base)))
np.testing.assert_allclose(circular_correlation(X,kernel).max(1),
                           circular_correlation(np.roll(X,5,axis=1),kernel).max(1))



## Étape 3
Décrire le rôle de chaque partition.


In [ ]:
# CELL 3 — Deux motifs de même somme, présents à des positions différentes.
L=32;K=3
motifs=np.array([[1.,1.,-1.],[1.,-1.,1.]])
def make_signals(n,positions,seed):
    r=np.random.default_rng(seed);y=r.integers(0,2,n)
    X=r.normal(0,.08,size=(n,L))
    for i in range(n):
        p=int(r.choice(positions))
        X[i,(p+np.arange(K))%L]+=motifs[y[i]]
    return X,y.reshape(-1,1).astype(float)

Xtr,ytr=make_signals(256,np.arange(0,12),10)
Xv,yv=make_signals(256,np.arange(0,12),20)
Xt,yt=make_signals(512,np.arange(0,12),30)
Xs,ys=make_signals(512,np.arange(16,28),40)
fig,ax=plt.subplots(1,2,figsize=(10,3))
for c in [0,1]:ax[c].plot(np.arange(K),motifs[c],'o-');ax[c].set(title=f'Motif classe {c}')
plt.tight_layout();plt.show()



## Étape 4
Compléter les gradients sans modifier le passage avant.


In [ ]:
# CELL 4 — CNN 1D : corrélation → ReLU → maximum global → logit.
def sigmoid(z):return np.exp(-np.logaddexp(0.,-z))
def init_model(kind,seed=0,width=8):
    r=np.random.default_rng(seed)
    fan=L if kind=='mlp' else K
    return {'W':r.normal(0,np.sqrt(2/fan),(fan,width)),
            'b':np.zeros((1,width)), 'V':r.normal(0,1/np.sqrt(width),(width,1)),
            'c':np.zeros((1,1))}

def network(X,p,kind):
    if kind=='mlp':
        u=X@p['W']+p['b'];h=np.maximum(u,0.)
        return h@p['V']+p['c'],(X,u,h)
    patches=np.stack([np.roll(X,-j,axis=1) for j in range(K)],axis=2)
    u=np.einsum('blk,kc->blc',patches,p['W'])+p['b']
    h=np.maximum(u,0.);where=np.argmax(h,axis=1)
    pooled=h.max(axis=1)
    return pooled@p['V']+p['c'],(patches,u,pooled,where)

def loss_grad(X,y,p,kind):
    z,cache=network(X,p,kind)
    loss=float(np.mean(np.logaddexp(0.,z)-y*z))
    d=(sigmoid(z)-y)/len(X)
    if kind=='mlp':
        original,u,h=cache;du=(d@p['V'].T)*(u>0)
        g={'W':original.T@du,'b':du.sum(0,keepdims=True),
           'V':h.T@d,'c':d.sum(0,keepdims=True)}
    else:
        patches,u,pooled,where=cache
        dh=np.zeros_like(u)
        dh[np.arange(len(X))[:,None],where,np.arange(u.shape[2])[None,:]]=d@p['V'].T
        du=dh*(u>0)
        g={'W':np.einsum('blk,blc->kc',patches,du),
           'b':du.sum(axis=(0,1))[None,:], 'V':pooled.T@d,
           'c':d.sum(0,keepdims=True)}
    return loss,g

def score(X,y,p,kind):
    z,_=network(X,p,kind)
    return float(np.mean(np.logaddexp(0.,z)-y*z)),float(((z>=0)==y).mean())



## Étape 5
Vérifier ReLU et max hors points non différentiables.


In [ ]:
# CELL 5 — Vérifier les gradients avant de comparer des modèles.
def check_gradients(p,objective,eps=1e-5):
    _,g=objective();errors={}
    for k,a in p.items():
        numeric=np.zeros_like(a)
        for idx in np.ndindex(a.shape):
            old=a[idx]
            try:
                a[idx]=old+eps;lp=objective()[0]
                a[idx]=old-eps;lm=objective()[0]
                numeric[idx]=(lp-lm)/(2*eps)
            finally:a[idx]=old
        errors[k]=float(np.max(np.abs(numeric-g[k])))
    return errors

for kind in ['mlp','cnn']:
    p=init_model(kind,seed=2,width=3)
    errors=check_gradients(p,lambda:loss_grad(Xtr[:4],ytr[:4],p,kind))
    print(kind,'erreurs absolues maximales :',errors)
    assert max(errors.values())<1e-6
# ReLU et max sont non différentiables aux égalités : éviter ces cas pour le check.



## Étape 6
Comparer coûts, graines et positions.


In [ ]:
# CELL 6 — Comparaison contrôlée, sans imposer le résultat.
def train(kind,seed=0,width=8,epochs=120):
    p=init_model(kind,seed,width)
    m={k:np.zeros_like(a) for k,a in p.items()};v={k:np.zeros_like(a) for k,a in p.items()}
    r=np.random.default_rng(100+seed);t=0;history=[]
    for epoch in range(epochs):
        order=r.permutation(len(Xtr))
        for start in range(0,len(order),32):
            ids=order[start:start+32];_,g=loss_grad(Xtr[ids],ytr[ids],p,kind);t+=1
            for k in p:
                m[k]=.9*m[k]+.1*g[k];v[k]=.999*v[k]+.001*g[k]**2
                p[k]-=.01*(m[k]/(1-.9**t))/(np.sqrt(v[k]/(1-.999**t))+1e-8)
        history.append(score(Xv,yv,p,kind)[0])
    return p,history

runs={}
for kind in ['mlp','cnn']:
    results=[]
    for seed in range(3):
        p,h=train(kind,seed);runs[kind,seed]=(p,h)
        results.append([score(Xt,yt,p,kind)[1],score(Xs,ys,p,kind)[1]])
    results=np.array(results)
    print(kind,'paramètres :',sum(a.size for a in p.values()),
          '; accuracy IID / positions décalées moyenne :',results.mean(0),
          '; écart-type :',results.std(0))
for kind in ['mlp','cnn']:plt.plot(runs[kind,0][1],label=kind)
plt.xlabel('Époque');plt.ylabel('NLL validation');plt.legend();plt.show()
# MLP largeur 8 : 273 paramètres ; CNN 8 canaux : 41 paramètres.
# Prolongement : CNN 54 canaux = 271 paramètres, proche du MLP, coût de calcul différent.



## Étape 7
Construire un cas où l’invariance empêche la bonne décision.


In [ ]:
# CELL 7 — Tester l'invariance ; puis supprimer de l'information utile.
p=runs['cnn',0][0]
z,_=network(Xt,p,'cnn');zr,_=network(np.roll(Xt,7,axis=1),p,'cnn')
np.testing.assert_allclose(z,zr,atol=1e-10,rtol=1e-10)
print('Écart maximal des logits CNN sous translation circulaire :',np.max(np.abs(z-zr)))
example=np.zeros((1,L));example[0,2]=1.
moved=np.roll(example,16,axis=1)
za,_=network(example,p,'cnn');zb,_=network(moved,p,'cnn')
np.testing.assert_allclose(za,zb,atol=1e-10)
print('Même représentation globale pour une impulsion à gauche ou à droite.')
# Si la cible demande gauche/droite, cette invariance empêche de résoudre cette paire.



## Étape 8
Retrouver les sommes sur le temps.


In [ ]:
# CELL 8 — Un petit RNN tanh et sa rétropropagation dans le temps.
def init_rnn(seed=3,inputs=2,hidden=4):
    r=np.random.default_rng(seed)
    return {'Wx':r.normal(0,.2,(inputs,hidden)), 'Wh':r.normal(0,.2,(hidden,hidden)),
            'b':np.zeros((1,hidden)), 'V':r.normal(0,.2,(hidden,1)), 'c':np.zeros((1,1))}

def rnn_loss_grad(X,y,p):
    B,T,_=X.shape;states=[np.zeros((B,p['Wh'].shape[0]))]
    for t in range(T):states.append(np.tanh(X[:,t]@p['Wx']+states[-1]@p['Wh']+p['b']))
    pred=states[-1]@p['V']+p['c'];err=(pred-y)/B
    loss=float(.5*np.mean((pred-y)**2));g={k:np.zeros_like(a) for k,a in p.items()}
    g['V']=states[-1].T@err;g['c']=err.sum(0,keepdims=True)
    dh=err@p['V'].T
    for t in reversed(range(T)):
        du=dh*(1-states[t+1]**2)
        g['Wx']+=X[:,t].T@du;g['Wh']+=states[t].T@du;g['b']+=du.sum(0,keepdims=True)
        dh=du@p['Wh'].T
    return loss,g

r=np.random.default_rng(4);seq=r.normal(size=(3,6,2));target=r.normal(size=(3,1))
p=init_rnn();errors=check_gradients(p,lambda:rnn_loss_grad(seq,target,p))
print('BPTT, erreurs absolues maximales :',errors)
assert max(errors.values())<1e-6



## Étape 9
Distinguer le chemin direct d’une porte fixée et la Jacobienne complète.


In [ ]:
# CELL 9 — Longueur du chemin et mémoire ; les portes sont ici fixées.
T=np.arange(1,101)
for a in [.5,.9,1.,1.05]:
    # h_t=a*h_{t-1}+x_t : d h_T / d x_1 = a**(T-1).
    plt.semilogy(T,np.abs(a)**(T-1),label=f'a={a}')
plt.xlabel('Longueur T');plt.ylabel('|d h_T / d x_1|');plt.legend();plt.show()
for forget in [.5,.9,.99]:
    print('Chemin direct de cellule, f fixé :',forget,'; dérivée après 100 pas :',forget**100)
# Dans une LSTM complète, les portes dépendent des états : ce chemin n'est pas toute la Jacobienne.



## Étape 10
Distinguer étendue et nombre de sites connectés.


In [ ]:
# CELL 10 — Champ réceptif, stride et dilation : calculer l'étendue.
def receptive_field(layers):
    r,j=1,1;trace=[]
    for kernel,stride,dilation in layers:
        r+=(kernel-1)*dilation*j;j*=stride;trace.append((r,j))
    return trace

print('3x3 stride1 ; 3x3 stride2 ; 3x3 dilatation2 :',
      receptive_field([(3,1,1),(3,2,1),(3,1,2)]))
assert receptive_field([(3,1,1),(3,2,1),(3,1,2)])[-1]==(13,2)
# L'étendue ne garantit pas que chaque position dans l'intervalle est connectée.
